In [1]:
import os
from dotenv import load_dotenv

notebook_dir = os.path.dirname(os.path.abspath("__file__")) 
parent_dir = os.path.dirname(notebook_dir)
env_path = os.path.join(parent_dir, ".env")

load_dotenv(env_path)

True

## Arrival

### Clean the database
- Only keep useful fields
    - Confident knowing the definition of the fields
    - Helpful and insightful data


- FID
- FlightNumber
- AirlineIATA
- FDate
- DepartureAirportIATA (出發機場)
- SCHE_TYPE (航班狀態)
- Cancel (取消)
- AircraftType (機型)
- SeatCapacity (座位數)
- LoadCapacity (載運量)
- Cargo (是否為貨運)
- amhsATA (實際到達時間)
- ArrivalDateTime (表定到達時間)
- Reg (註冊編號)
- Passanger (旅客人數)
- Bay (停機位)
- RWYARR (跑道)
- logtime (紀錄時間)

In [2]:
import pandas as pd

In [14]:
arrival_df = pd.read_csv("arrival_bk.csv")
len(arrival_df.columns)

/tmp/ipykernel_1774491/4055908744.py:1: DtypeWarning: Columns (20,28,37) have mixed types. Specify dtype option on import or set low_memory=False.
  arrival_df = pd.read_csv("arrival_bk.csv")


44

In [13]:
# Updated confident fields based on new field mapping (17 fields for Arrival)
confident_fields = [
    "FID", "FDate", "AirlineIATA", "FlightNumber", "DepartureAirportIATA", 
    "SCHE_TYPE", "Cancel", "AircraftType", "SeatCapacity", "LoadCapacity", 
    "Cargo", "amhsATA", "ArrivalDateTime", "Reg", "Passanger", "Bay", 
    "RWYARR", "logtime"
]

In [21]:
# pip install pandas pymongo python-dotenv
import os
from pathlib import Path
from typing import Iterable, Dict, Any, Optional

import numpy as np
import pandas as pd
from pymongo import MongoClient, ASCENDING
from pymongo.errors import BulkWriteError
from dotenv import load_dotenv

import numpy as np
import pandas as pd
from pymongo import MongoClient, ASCENDING, UpdateOne, InsertOne
from pymongo.errors import BulkWriteError
from dotenv import load_dotenv
from datetime import datetime

# ----------- Config -----------
MONGO_URI = os.getenv("MONGODB_URL")
COLL_NAME = os.getenv("TEST_ARRIVAL_COLLECTTION_NAME", "test_arrival")
DB_NAME = os.getenv("DATABASE_NAME", "KHH_airport_demo")
CSV_PATH = "arrival_bk.csv"
UPSERT_KEY = "FID"        # unique key for upserts
CHUNKSIZE = 5000
BATCH_WRITE = 1000

# Updated columns to keep from arrival_bk.csv based on new field mapping (17 fields)
# Temporarily include additional ATA columns for preprocessing
KEEP_COLS = [
    "FID", "FDate", "AirlineIATA", "FlightNumber", "DepartureAirportIATA", 
    "SCHE_TYPE", "Cancel", "AircraftType", "SeatCapacity", "LoadCapacity", 
    "Cargo", "amhsATA", "ArrivalDateTime", "Reg", "Passanger", "Bay", 
    "RWYARR", "logtime",
    # Temporary columns for ATA consolidation (will be dropped after processing)
    "aissATA", "ritATA"
]

# Datetime-like columns in arrival_bk.csv (only meaningful timestamps)
DATE_COLS = [
    "FDate","ArrivalDateTime","logtime", "amhsATA",
    "aissATA", "ritATA"  # Additional ATA columns for preprocessing
]

# Integer-like columns (nullable)
INT_COLS = [
    "FID","FlightNumber","SeatCapacity","LoadCapacity","Bay","Passanger","Cancel", "Cargo", "RWYARR"
]

# String/categorical columns
STR_COLS = [
    "AirlineIATA","DepartureAirportIATA","SCHE_TYPE","AircraftType","Reg"
]

NA_VALUES = ["NULL", "", " ", "NA", "N/A", None, "0000-00-00 00:00:00"]


def parse_chunk(df: pd.DataFrame) -> pd.DataFrame:
    # Reduce to keep-list if present
    existing = [c for c in KEEP_COLS if c in df.columns]
    df = df[existing].copy()

    # Normalize NA markers first
    df = df.replace({np.nan: None})
    for c in df.columns:
        df[c] = df[c].replace(NA_VALUES, None)

    # Parse dates -> pandas Timestamp
    for col in DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # Cast integers (nullable)
    for col in INT_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    # Ensure strings (strip)
    for col in STR_COLS:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()

    # Replace amhsATA with first non-NA ATA value from multiple sources
    if all(col in df.columns for col in ['amhsATA', 'aissATA', 'ritATA']):
        # Use your simple approach: backfill to get first non-null value
        actual = df[['amhsATA', 'aissATA', 'ritATA']].bfill(axis=1).iloc[:, 0]
        df['amhsATA'] = actual
        
        # Drop the temporary ATA columns since we've consolidated them
        df = df.drop(['aissATA', 'ritATA'], axis=1)

    # Set Cancel to 0 if ArrivalDateTime exists
    if 'Cancel' in df.columns and 'ArrivalDateTime' in df.columns:
        df.loc[df['ArrivalDateTime'].notna(), 'Cancel'] = 0

    # Final NA to None for Mongo
    df = df.replace({pd.NA: None, np.nan: None})
    return df

def to_python_types(doc: Dict[str, Any]) -> Dict[str, Any]:
    """Ensure values are Python-native for PyMongo (e.g., datetimes, ints)."""
    out = {}
    for k, v in doc.items():
        if v is None:
            out[k] = None
        elif isinstance(v, (np.integer,)):
            out[k] = int(v)
        elif isinstance(v, pd.Timestamp):
            out[k] = v.to_pydatetime()
        else:
            out[k] = v
    return out

def make_bulk_ops(records: Iterable[Dict[str, Any]], upsert_key: Optional[str]) -> list:
    ops = []
    for raw in records:
        doc = to_python_types(raw)
        if upsert_key and doc.get(upsert_key) is not None:
            ops.append(UpdateOne({upsert_key: doc[upsert_key]}, {"$set": doc}, upsert=True))
        else:
            ops.append(InsertOne(doc))
    return ops

def ensure_indexes(coll):
    # Unique on FID
    try:
        coll.create_index([("FID", ASCENDING)], unique=True, name="uniq_fid")
    except Exception:
        pass

    coll.create_index([("ArrivalDateTime", ASCENDING)], name="by_arrival_dt")
    coll.create_index([("AirlineIATA", ASCENDING), ("ArrivalDateTime", ASCENDING)], name="by_airline_arrival")
    coll.create_index([("Bay", ASCENDING)], name="by_bay")
    coll.create_index([("RWYARR", ASCENDING)], name="by_runway")

def load_csv_to_mongo(
    csv_path: str,
    mongo_uri: str,
    db_name: str,
    coll_name: str,
    upsert_key: Optional[str] = "FID",
    chunksize: int = 5000,
    batch_write: int = 1000,
    date_filter_col: Optional[str] = None,
    date_min: Optional[str] = None,
    date_max: Optional[str] = None,
):
    client = MongoClient(mongo_uri)
    coll = client[db_name][coll_name]
    ensure_indexes(coll)

    reader = pd.read_csv(
        csv_path,
        chunksize=chunksize,
        na_values=NA_VALUES,
        keep_default_na=True
    )

    total_rows = 0
    total_upserts = 0
    total_inserted = 0
    total_modified = 0

    for chunk in reader:
        chunk = parse_chunk(chunk)

        # Apply date filter before writing (if configured)
        if date_filter_col and date_filter_col in chunk.columns:
            if date_min:
                chunk = chunk[chunk[date_filter_col] >= pd.to_datetime(date_min)]
            if date_max:
                chunk = chunk[chunk[date_filter_col] <= pd.to_datetime(date_max)]

        records = chunk.to_dict(orient="records")
        total_rows += len(records)

        sub = []
        for rec in records:
            sub.append(rec)
            if len(sub) >= batch_write:
                ops = make_bulk_ops(sub, upsert_key)
                if ops:
                    try:
                        res = coll.bulk_write(ops, ordered=False)
                        total_upserts += len(getattr(res, "upserted_ids", {}) or {})
                        total_inserted += res.inserted_count
                        total_modified += res.modified_count
                    except BulkWriteError as bwe:
                        print("BulkWriteError (sample):", bwe.details.get("writeErrors", [])[:3])
                sub = []

        if sub:
            ops = make_bulk_ops(sub, upsert_key)
            if ops:
                try:
                    res = coll.bulk_write(ops, ordered=False)
                    total_upserts += len(getattr(res, "upserted_ids", {}) or {})
                    total_inserted += res.inserted_count
                    total_modified += res.modified_count
                except BulkWriteError as bwe:
                    print("BulkWriteError (sample):", bwe.details.get("writeErrors", [])[:3])

    print(f"Processed rows: {total_rows}")
    print(f"Inserted: {total_inserted}, Upserted: {total_upserts}, Modified: {total_modified}")


In [22]:
# Clear MongoDB and reinsert data with updated field mapping
print("=== CLEARING MONGODB AND REINSERTING DATA WITH NEW FIELD MAPPING ===")

# Clear arrival collection
arrival_collection = MongoClient(MONGO_URI)[DB_NAME][COLL_NAME]
arrival_collection.delete_many({})
print("✅ Cleared arrival collection")

# Reinsert arrival data with new field mapping
print("Reinserting arrival data with 18 required fields...")
load_csv_to_mongo(
    csv_path=CSV_PATH,
    mongo_uri=MONGO_URI,
    db_name=DB_NAME,
    coll_name=COLL_NAME,
    upsert_key=UPSERT_KEY,
    chunksize=CHUNKSIZE,
    batch_write=BATCH_WRITE,
    date_filter_col="ArrivalDateTime",
    date_min=None, # Set to None if wants to insert all data
    date_max=None
)

=== CLEARING MONGODB AND REINSERTING DATA WITH NEW FIELD MAPPING ===
✅ Cleared arrival collection
Reinserting arrival data with 18 required fields...
Processed rows: 29083
Inserted: 0, Upserted: 29083, Modified: 0


In [23]:
# Read from the database and double check whether the column length matches the expected length
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLL_NAME]
df = pd.DataFrame(list(collection.find({},{"_id":0})))
print(len(df.columns)==len(confident_fields), len(df.columns), len(confident_fields))

print(list(df.columns))
sorted(list(df.columns)) == sorted(confident_fields)

True 18 18
['FID', 'AircraftType', 'AirlineIATA', 'ArrivalDateTime', 'Bay', 'Cancel', 'Cargo', 'DepartureAirportIATA', 'FDate', 'FlightNumber', 'LoadCapacity', 'Passanger', 'RWYARR', 'Reg', 'SCHE_TYPE', 'SeatCapacity', 'amhsATA', 'logtime']


True

## Departure

Departure fields (18 total):

- FID
- FlightNumber
- AirlineIATA
- FDate
- ArrivalAirportIATA (到達機場)
- SCHE_TYPE (航班狀態)
- Cancel (取消)
- AircraftType (機型)
- SeatCapacity (座位數)
- LoadCapacity (載運量)
- Cargo (是否為貨運)
- amhsATD (實際出發時間)
- DepartureDateTime (表定出發時間)
- Reg (註冊編號)
- Passanger (旅客人數)
- Bay (停機位)
- RWYDEP (跑道)
- logtime (紀錄時間)

In [35]:
import pandas as pd

In [36]:
departure_csv = pd.read_csv("depart_bk.csv")
len(departure_csv.columns)

/tmp/ipykernel_1774491/931790842.py:1: DtypeWarning: Columns (35) have mixed types. Specify dtype option on import or set low_memory=False.
  departure_csv = pd.read_csv("depart_bk.csv")


39

In [37]:
# REFINED DEPARTURE FIELD CATEGORIZATION BASED ON DATA ANALYSIS

# Updated confident fields based on new field mapping (17 fields for Departure)
confident_fields = [
    "FID","FDate","AirlineIATA","FlightNumber","ArrivalAirportIATA",
    "SCHE_TYPE","Cancel","AircraftType","SeatCapacity","LoadCapacity",
    "Cargo","amhsATD","DepartureDateTime","Reg","Passanger","Bay",
    "RWYDEP","logtime",
]
len(confident_fields)

18

In [32]:
# ----------- Departure Configuration -----------
DEPART_COLL_NAME = "test_departure"
DEPART_CSV_PATH = "depart_bk.csv"

# Updated DEPART_KEEP_COLS to include temporary ATD columns
UPDATED_DEPART_KEEP_COLS = [
    "FID","FDate","AirlineIATA","FlightNumber","ArrivalAirportIATA",
    "SCHE_TYPE","Cancel","AircraftType","SeatCapacity","LoadCapacity",
    "Cargo","DepartureDateTime","Reg","Passanger","Bay",
    "RWYDEP","logtime","amhsATD", 
    # Temporary columns for ATD consolidation (will be dropped after processing)
    "aissATD", "ritATD"
]

# Updated DEPART_DATE_COLS to include temporary ATD columns
UPDATED_DEPART_DATE_COLS = [
    "FDate", "DepartureDateTime", "logtime",
    "amhsATD", "aissATD", "ritATD"  # Additional ATD columns for preprocessing
]

# Integer-like columns (nullable)
DEPART_INT_COLS = [
    "FID", "FlightNumber", "SeatCapacity", "Bay", "Passanger", "LoadCapacity", "Cancel", "Cargo", "RWYDEP"
]

# String/categorical columns
DEPART_STR_COLS = [
    "AirlineIATA", "AircraftType", "Reg", "ArrivalAirportIATA", "SCHE_TYPE"
]

NA_VALUES = ["NULL", "", " ", "NA", "N/A", None, "0000-00-00 00:00:00"]

print("Departure configuration loaded")


Departure configuration loaded


In [33]:
# Make sure these are in scope
# import numpy as np
# from pymongo import ASCENDING

# Updated departure parse function with ATD consolidation
def parse_departure_chunk(df: pd.DataFrame) -> pd.DataFrame:
    """Parse departure data chunk with ATD consolidation (same logic as arrival)"""
    # Reduce to keep-list if present
    existing = [c for c in UPDATED_DEPART_KEEP_COLS if c in df.columns]
    df = df[existing].copy()

    # Normalize NA markers first
    df = df.replace({np.nan: None})
    for c in df.columns:
        df[c] = df[c].replace(NA_VALUES, None)

    # Parse dates -> pandas Timestamp
    for col in UPDATED_DEPART_DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # Cast integers (nullable)
    for col in DEPART_INT_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    # Ensure strings (strip)
    for col in DEPART_STR_COLS:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()

    # Note: We're consolidating ATD (Actual Time of Departure) into amhsATD
    if all(col in df.columns for col in ['amhsATD', 'aissATD', 'ritATD']):
        # Use the same simple approach: backfill to get first non-null value
        actual = df[['amhsATD', 'aissATD', 'ritATD']].bfill(axis=1).iloc[:, 0]
        df['amhsATD'] = actual
        
        # Drop the temporary ATD columns since we've consolidated them
        df = df.drop(['aissATD', 'ritATD'], axis=1)

    # Set Cancel to 0 if DepartureDateTime exists
    if 'Cancel' in df.columns and 'DepartureDateTime' in df.columns:
        df.loc[df['DepartureDateTime'].notna(), 'Cancel'] = 0
    
    # Final NA to None for Mongo
    df = df.replace({pd.NA: None, np.nan: None})
        
    return df

print("Updated departure parse function created!")



def ensure_departure_indexes(coll):
    """Create indexes optimized for departure queries (only on kept fields)."""
    # Unique on FID
    try:
        coll.create_index([("FID", ASCENDING)], unique=True, name="uniq_fid")
    except Exception:
        pass

    # Departure datetime for time-range queries
    coll.create_index([("DepartureDateTime", ASCENDING)], name="by_departure_dt")

    # Airline + time (common filter)
    coll.create_index(
        [("AirlineIATA", ASCENDING), ("DepartureDateTime", ASCENDING)],
        name="by_airline_departure"
    )

    # Destination
    coll.create_index([("ArrivalAirportIATA", ASCENDING)], name="by_destination")

    # Ops surfaces
    coll.create_index([("Bay", ASCENDING)], name="by_bay")
    coll.create_index([("RWYDEP", ASCENDING)], name="by_runway")


print("Departure functions defined")


Updated departure parse function created!
Departure functions defined


In [38]:
def load_departure_to_mongo(
    csv_path: str,
    mongo_uri: str,
    db_name: str,
    coll_name: str,
    upsert_key: Optional[str] = "FID",
    chunksize: int = 5000,
    batch_write: int = 1000,
    date_filter_col: Optional[str] = None,
    date_min: Optional[str] = None,
    date_max: Optional[str] = None,
):
    """Load departure data with departure-specific parsing and indexing"""
    client = MongoClient(mongo_uri)
    coll = client[db_name][coll_name]
    ensure_departure_indexes(coll)

    reader = pd.read_csv(
        csv_path,
        chunksize=chunksize,
        na_values=NA_VALUES,
        keep_default_na=True
    )

    total_rows = 0
    total_upserts = 0
    total_inserted = 0
    total_modified = 0

    for chunk in reader:
        chunk = parse_departure_chunk(chunk)

        # Apply date filter before writing (if configured)
        if date_filter_col and date_filter_col in chunk.columns:
            if date_min:
                chunk = chunk[chunk[date_filter_col] >= pd.to_datetime(date_min)]
            if date_max:
                chunk = chunk[chunk[date_filter_col] <= pd.to_datetime(date_max)]

        records = chunk.to_dict(orient="records")
        total_rows += len(records)

        sub = []
        for rec in records:
            sub.append(rec)
            if len(sub) >= batch_write:
                ops = make_bulk_ops(sub, upsert_key)
                if ops:
                    try:
                        res = coll.bulk_write(ops, ordered=False)
                        total_upserts += len(getattr(res, "upserted_ids", {}) or {})
                        total_inserted += res.inserted_count
                        total_modified += res.modified_count
                    except BulkWriteError as bwe:
                        print("BulkWriteError (sample):", bwe.details.get("writeErrors", [])[:3])
                sub = []

        if sub:
            ops = make_bulk_ops(sub, upsert_key)
            if ops:
                try:
                    res = coll.bulk_write(ops, ordered=False)
                    total_upserts += len(getattr(res, "upserted_ids", {}) or {})
                    total_inserted += res.inserted_count
                    total_modified += res.modified_count
                except BulkWriteError as bwe:
                    print("BulkWriteError (sample):", bwe.details.get("writeErrors", [])[:3])

    print(f"Processed departure rows: {total_rows}")
    print(f"Inserted: {total_inserted}, Upserted: {total_upserts}, Modified: {total_modified}")

print("Departure loader function defined")

Departure loader function defined


In [40]:
# Load departure data into test_departure collection using optimized departure loader
print("Loading departure data with optimized field selection...")

# Ensure the departure database to be empty first (following arrival pattern)
departure_collection = MongoClient(MONGO_URI)[DB_NAME][DEPART_COLL_NAME]
departure_collection.delete_many({})
print("Cleared existing departure data")

load_departure_to_mongo(
    csv_path=DEPART_CSV_PATH,
    mongo_uri=MONGO_URI,
    db_name=DB_NAME,
    coll_name=DEPART_COLL_NAME,
    upsert_key="FID",
    chunksize=5000,
    batch_write=1000,
    date_filter_col="DepartureDateTime",
    date_min=None, # Set to None if wants to insert all data
    date_max=None
)

Loading departure data with optimized field selection...
Cleared existing departure data
Processed departure rows: 29026
Inserted: 0, Upserted: 29026, Modified: 0


In [41]:
# Double check the data
collection = MongoClient(MONGO_URI)[DB_NAME][DEPART_COLL_NAME]
departure_df_mongo = pd.DataFrame(list(collection.find({}, {"_id": 0})))

print(list(departure_df_mongo.columns))
sorted(list(departure_df_mongo.columns)) == sorted(confident_fields)


['FID', 'AircraftType', 'AirlineIATA', 'ArrivalAirportIATA', 'Bay', 'Cancel', 'Cargo', 'DepartureDateTime', 'FDate', 'FlightNumber', 'LoadCapacity', 'Passanger', 'RWYDEP', 'Reg', 'SCHE_TYPE', 'SeatCapacity', 'amhsATD', 'logtime']


True